# Phase 3-4: Attack Effectiveness

Measure how much each attack degrades YOLOv8 detection.

In [ ]:
import os
import sys
import yaml
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

IN_COLAB = 'google.colab' in sys.modules
print(f"Colab: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    !pip install ultralytics -q
    from google.colab import drive
    drive.mount('/content/drive')

from ultralytics import YOLO
import torch
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# paths
if IN_COLAB:
    BASE = "/content/drive/MyDrive/Colab Notebooks/data"
    STAGING = "/content/staged_data"
else:
    BASE = "/Users/tyreecruse/Desktop/CS230/Project/Data"
    STAGING = BASE

MODEL_PATH = f"{BASE}/training/results from training/weights/best.pt"
ZIPS_DIR = f"{BASE}/analysis/zips"
RESULTS_DIR = f"{BASE}/analysis/results"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(STAGING, exist_ok=True)

ATTACKS = [
    "fgsm_030", "fgsm_045", "fgsm_060", "fgsm_075", "fgsm_090", "fgsm_105",
    "gaussian_010", "gaussian_050", "gaussian_150", "gaussian_200", "gaussian_250",
    "patches",
]

print(f"Model: {MODEL_PATH}")

In [ ]:
# load model
print("Loading model...")
model = YOLO(MODEL_PATH)
print(f"Classes: {model.model.names}")

In [ ]:
# evaluate clean baseline
print("\n" + "="*50)
print("CLEAN BASELINE")
print("="*50)

# unzip clean if needed
cleanZip = f"{ZIPS_DIR}/clean.zip"
cleanPath = f"{STAGING}/clean"
if not os.path.exists(cleanPath):
    print("Unzipping clean...")
    with zipfile.ZipFile(cleanZip, 'r') as zf:
        zf.extractall(STAGING)

# find actual data dir (handle nested structures)
dataDir = cleanPath
if os.path.exists(f"{cleanPath}/images"):
    dataDir = cleanPath
elif os.path.exists(f"{cleanPath}/clean/images"):
    dataDir = f"{cleanPath}/clean"
print(f"Data dir: {dataDir}")

# create data.yaml
# note: nc=2 because dataset has mixed class indices (0 and 1) from merging roboflow exports
yamlPath = f"{STAGING}/clean_data.yaml"
yamlData = {
    'path': dataDir,
    'train': 'images',
    'val': 'images',
    'test': 'images',
    'nc': 2,
    'names': ['tank', 'tank']
}
with open(yamlPath, 'w') as f:
    yaml.dump(yamlData, f)

# run validation
print("Running validation...")
cleanResults = model.val(data=yamlPath, conf=0.25, iou=0.5, verbose=False, plots=False)

cleanMap = cleanResults.box.map50
cleanRecall = cleanResults.box.mr
cleanPrec = cleanResults.box.mp

print(f"\nmAP@0.5:   {cleanMap:.3f}")
print(f"Recall:    {cleanRecall:.3f}")
print(f"Precision: {cleanPrec:.3f}")

In [ ]:
# evaluate each attack
results = []

for attackName in ATTACKS:
    print(f"\n{'-'*50}")
    print(f"Evaluating: {attackName}")
    print('-'*50)
    
    try:
        # unzip if needed
        zipPath = f"{ZIPS_DIR}/{attackName}.zip"
        stagePath = f"{STAGING}/{attackName}"
        
        if not os.path.exists(stagePath):
            print("Unzipping...")
            with zipfile.ZipFile(zipPath, 'r') as zf:
                zf.extractall(STAGING)
        
        # find data dir
        dataDir = stagePath
        if os.path.exists(f"{stagePath}/images"):
            dataDir = stagePath
        elif os.path.exists(f"{stagePath}/{attackName}/images"):
            dataDir = f"{stagePath}/{attackName}"
        
        # create yaml
        yamlPath = f"{STAGING}/{attackName}_data.yaml"
        yamlData = {
            'path': dataDir,
            'train': 'images',
            'val': 'images',
            'test': 'images',
            'nc': 2,
            'names': ['tank', 'tank']
        }
        with open(yamlPath, 'w') as f:
            yaml.dump(yamlData, f)
        
        # run validation
        attackResults = model.val(data=yamlPath, conf=0.25, iou=0.5, verbose=False, plots=False)
        
        attackMap = attackResults.box.map50
        attackRecall = attackResults.box.mr
        
        # compute drops
        mapDrop = cleanMap - attackMap
        mapDropPct = (mapDrop / cleanMap) * 100
        recallDrop = cleanRecall - attackRecall
        recallDropPct = (recallDrop / cleanRecall) * 100
        
        # parse attack type
        if attackName.startswith('fgsm'):
            attackType = 'FGSM'
            strength = int(attackName.split('_')[1]) / 1000
        elif attackName.startswith('gaussian'):
            attackType = 'Gaussian'
            strength = int(attackName.split('_')[1]) / 1000
        else:
            attackType = 'Patch'
            strength = None
        
        results.append({
            'attack': attackName,
            'attackType': attackType,
            'strength': strength,
            'mAP50': attackMap,
            'recall': attackRecall,
            'mAP50Drop': mapDrop,
            'mAP50DropPct': mapDropPct,
            'recallDropPct': recallDropPct,
        })
        
        print(f"mAP@0.5: {attackMap:.3f} (↓{mapDropPct:.1f}%)")
        print(f"Recall:  {attackRecall:.3f} (↓{recallDropPct:.1f}%)")
        
    except Exception as e:
        print(f"Error: {e}")

print(f"\nEvaluated {len(results)} attacks")

In [ ]:
# display results
df = pd.DataFrame(results)

print("\n" + "="*70)
print("ATTACK EFFECTIVENESS")
print("="*70)
print(f"Baseline: mAP@0.5={cleanMap:.3f}, Recall={cleanRecall:.3f}")
print()
print(df[['attackType', 'strength', 'mAP50', 'mAP50DropPct', 'recall', 'recallDropPct']].round(3).to_string(index=False))

# save
df.to_csv(f"{RESULTS_DIR}/attack_effectiveness.csv", index=False)
print(f"\nSaved to {RESULTS_DIR}/attack_effectiveness.csv")

In [ ]:
# plot effectiveness vs strength
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# FGSM
fgsm = df[df['attackType'] == 'FGSM'].sort_values('strength')
axes[0].plot(fgsm['strength'], fgsm['mAP50DropPct'], 'bo-', linewidth=2, markersize=8, label='mAP drop')
axes[0].plot(fgsm['strength'], fgsm['recallDropPct'], 'rs-', linewidth=2, markersize=8, label='Recall drop')
axes[0].set_xlabel('FGSM ε')
axes[0].set_ylabel('Performance Drop (%)')
axes[0].set_title('FGSM Effectiveness')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(bottom=0)

# Gaussian
gauss = df[df['attackType'] == 'Gaussian'].sort_values('strength')
axes[1].plot(gauss['strength'], gauss['mAP50DropPct'], 'go-', linewidth=2, markersize=8, label='mAP drop')
axes[1].plot(gauss['strength'], gauss['recallDropPct'], 'ms-', linewidth=2, markersize=8, label='Recall drop')
axes[1].set_xlabel('Gaussian σ')
axes[1].set_ylabel('Performance Drop (%)')
axes[1].set_title('Gaussian Effectiveness')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/attack_effectiveness.png", dpi=150)
plt.show()

In [ ]:
# tradeoff analysis (if detection results exist)
detPath = f"{RESULTS_DIR}/detection_performance.csv"

if os.path.exists(detPath):
    detDf = pd.read_csv(detPath)
    
    # merge
    merged = []
    for _, row in df.iterrows():
        match = detDf[detDf['Attack'] == row['attack']]
        if len(match) > 0:
            merged.append({
                'attack': row['attack'],
                'attackType': row['attackType'],
                'strength': row['strength'],
                'detectionAuc': match.iloc[0]['ROC-AUC'],
                'attackSuccess': row['recallDropPct'] / 100,
            })
    
    if merged:
        tradeoffDf = pd.DataFrame(merged)
        
        # plot
        fig, ax = plt.subplots(figsize=(10, 8))
        
        colors = {'FGSM': 'blue', 'Gaussian': 'green', 'Patch': 'red'}
        
        for atype in tradeoffDf['attackType'].unique():
            subset = tradeoffDf[tradeoffDf['attackType'] == atype]
            ax.scatter(subset['detectionAuc'], subset['attackSuccess'], 
                      c=colors[atype], s=100, label=atype, alpha=0.7)
            
            for _, r in subset.iterrows():
                lbl = f"{r['strength']:.3f}" if r['strength'] else 'Patch'
                ax.annotate(lbl, (r['detectionAuc'], r['attackSuccess']),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax.axhspan(0, 0.1, alpha=0.1, color='green')  # weak attacks
        ax.axvspan(0.9, 1.0, alpha=0.1, color='blue')  # easily detected
        
        ax.set_xlabel('Detection AUC (higher = more detectable)')
        ax.set_ylabel('Attack Success Rate')
        ax.set_title('Detectability vs Effectiveness')
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_xlim([0.4, 1.02])
        ax.set_ylim([-0.05, 1.0])
        
        plt.tight_layout()
        plt.savefig(f"{RESULTS_DIR}/detectability_vs_effectiveness.png", dpi=150)
        plt.show()
else:
    print("Detection results not found - run Phase 3-3 first")

In [ ]:
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

print("\nFGSM:")
for _, r in fgsm.iterrows():
    status = "Effective" if r['mAP50DropPct'] > 20 else "Moderate" if r['mAP50DropPct'] > 5 else "Weak"
    print(f"  ε={r['strength']:.3f}: mAP↓{r['mAP50DropPct']:.1f}%, Recall↓{r['recallDropPct']:.1f}% - {status}")

print("\nGaussian:")
for _, r in gauss.iterrows():
    status = "Effective" if r['mAP50DropPct'] > 20 else "Moderate" if r['mAP50DropPct'] > 5 else "Weak"
    print(f"  σ={r['strength']:.3f}: mAP↓{r['mAP50DropPct']:.1f}%, Recall↓{r['recallDropPct']:.1f}% - {status}")

patch = df[df['attackType'] == 'Patch']
if len(patch) > 0:
    r = patch.iloc[0]
    status = "Effective" if r['mAP50DropPct'] > 20 else "Moderate" if r['mAP50DropPct'] > 5 else "Weak"
    print(f"\nPatches: mAP↓{r['mAP50DropPct']:.1f}%, Recall↓{r['recallDropPct']:.1f}% - {status}")

print("\nDone!")

In [ ]:
# cleanup staging (uncomment to run)
# shutil.rmtree(STAGING)
# print("Cleaned up staging")